<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/NSMC_SentencePiece_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 네이버 영화리뷰 감정 분석 문제에 SentencePiece 적용해 보기

```text
- 네이버 영화리뷰 감정 분석 코퍼스에 SentencePiece를 적용시킨 모델 학습하기
- 학습된 모델로 sp_tokenize() 메소드 구현하기
- 구현된 토크나이저를 적용하여 네이버 영화리뷰 감정 분석 모델을 재학습하기
- KoNLPy 형태소 분석기를 사용한 모델과 성능 비교하기
- SentencePiece 모델의 model_type, vocab_size 등을 변경해 가면서 성능 개선 여부 확인하기  
```
```text
- SentencePiece를 이용하여 모델을 만들기까지의 과정이 정상적으로 진행되었는가?  
 :코퍼스 분석, 전처리, SentencePiece 적용, 토크나이저 구현 및 동작이 빠짐없이 진행되었는지를 봅니다.
- SentencePiece를 통해 만든 Tokenizer가 자연어처리 모델과 결합하여 동작하는가?  
 :SentencePiece 토크나이저가 적용된 Text Classifier 모델이 정상적으로 수렴하여 80% 이상의 test accuracy가 확인되었다면 성공입니다.
- SentencePiece의 성능을 다각도로 비교분석하였는가?  
 :SentencePiece 토크나이저를 활용했을 때의 성능을 다른 토크나이저 혹은 SentencePiece의 다른 옵션의 경우와 비교하여 분석을 체계적으로 진행하였는지 봅니다.

In [151]:
import sys

try:
    import sentencepiece
except ImportError:
    !{sys.executable} -m pip install -q sentencepiece==0.2.2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

SEED = 42
torch.manual_seed(SEED)

# Colab GPU가 있으면 CUDA, 없으면 CPU를 사용합니다.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_DIR = Path("/content/sentencepiece_basic") if Path("/content").exists() else Path("/tmp/sentencepiece_basic")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

MAX_LENGTH = 80
BATCH_SIZE = 512
EPOCHS = 3

print("device:", DEVICE)
print("work directory:", WORK_DIR)


device: cuda
work directory: /content/sentencepiece_basic


### 1. NSMC 데이터 불러오기


In [152]:
NSMC_COMMIT = "cc0670e872d4ac27bfe36c87456783004b39ef6c"
TRAIN_URL = f"https://raw.githubusercontent.com/e9t/nsmc/{NSMC_COMMIT}/ratings_train.txt"
TEST_URL = f"https://raw.githubusercontent.com/e9t/nsmc/{NSMC_COMMIT}/ratings_test.txt"

train_data = pd.read_table(TRAIN_URL)
test_data = pd.read_table(TEST_URL)

# document가 비어 있는 행과 같은 문장이 반복된 행을 제거합니다.
train_data = train_data.dropna(subset=["document", "label"])
test_data = test_data.dropna(subset=["document", "label"])
train_data = train_data.drop_duplicates(subset=["document"])
test_data = test_data.drop_duplicates(subset=["document"])

# train에 있던 문장이 test에도 있으면 test에서 제외합니다.
test_data = test_data[~test_data["document"].isin(train_data["document"])]

# train 중 5,000개를 validation으로 사용합니다.
validation_data = train_data.sample(n=5000, random_state=SEED)
train_data = train_data.drop(validation_data.index)

train_data = train_data.reset_index(drop=True)
validation_data = validation_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print("train:", len(train_data))
print("validation:", len(validation_data))
print("test:", len(test_data))

train: 141182
validation: 5000
test: 48361


In [153]:
display(train_data.head(3))
print("train 긍정 비율:", round(train_data["label"].mean(), 3))
print("validation 긍정 비율:", round(validation_data["label"].mean(), 3))
print("test 긍정 비율:", round(test_data["label"].mean(), 3))

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0


train 긍정 비율: 0.498
validation 긍정 비율: 0.506
test 긍정 비율: 0.503


### 2. Unigram SentencePiece 학습

- `filtered_corpus`: 앞 단계에서 정제한 문장 목록
- `temp_file`: SentencePiece가 읽을 text 파일
- `vocab_size = 8000`: 만들 token 조각 수
- `model_type`을 쓰지 않으면 LMS 설명처럼 Unigram이 기본값

In [154]:
filtered_corpus = train_data["document"].tolist()

import sentencepiece as spm
import os
temp_file = 'korean-english-park.train.ko.temp'

vocab_size = 8000

with open(temp_file, 'w', encoding='utf-8') as f:
    for row in filtered_corpus:   # 이전에 나왔던 정제했던 corpus를 활용해서 진행해야 합니다.
        f.write(str(row) + '\n')

spm.SentencePieceTrainer.Train(
    '--input={} --model_prefix=korean_spm --vocab_size={} --pad_id=3 --minloglevel=2'.format(
        temp_file, vocab_size
    )
)
# 위 Train에서 --model_type=unigram이 default(기본값)입니다.

!ls -l korean_spm*

-rw-r--r-- 1 root root 802540 Sep  2 21:35 korean_spm_bpe.model
-rw-r--r-- 1 root root 521315 Sep  2 21:35 korean_spm_bpe.vocab
-rw-r--r-- 1 root root 377698 Sep  2 21:38 korean_spm.model
-rw-r--r-- 1 root root 144719 Sep  2 21:38 korean_spm.vocab


### 3. Encode/Decode 확인

- `EncodeAsIds()`: 문장을 숫자 ID 목록으로 변경
- `SampleEncodeAsPieces()`: 문장을 사람이 읽을 수 있는 subword(부분단어)로 표시
- `DecodeIds()`: ID 목록을 다시 문장으로 복원

In [155]:
s = spm.SentencePieceProcessor()
s.Load('korean_spm.model')

# SentencePiece를 활용한 sentence -> encoding
tokensIDs = s.EncodeAsIds('아버지가방에들어가신다.')
print(tokensIDs)

# SentencePiece를 활용한 sentence -> encoded pieces
print(s.SampleEncodeAsPieces('아버지가방에들어가신다.', -1, 0.1))

# SentencePiece를 활용한 encoding -> sentence 복원
print(s.DecodeIds(tokensIDs))

[1406, 12, 387, 16, 1317, 12, 131, 19, 5]
['▁', '아버지', '가', '방', '에', '들', '어', '가', '신', '다', '.']
아버지가방에들어가신다.


### 4.`sp_tokenize()` 구현

1. `EncodeAsIds()` 결과인 Python `list`를 `torch.Tensor`로 바꿉니다.
2. padding에 `<unk>`의 ID인 `0`이 아니라 `s.pad_id()`인 `3`을 사용합니다.


- `[:MAX_LENGTH]`: 아주 긴 리뷰를 앞의 80 token까지만 사용해 메모리를 제한합니다.
- `encoding='utf-8'`: 한국어 vocab 파일을 같은 인코딩으로 읽습니다.
- `vocab_path`: 뒤에서 BPE vocabulary도 같은 함수로 읽습니다.

In [156]:
def sp_tokenize(s, corpus, vocab_path="./korean_spm.vocab"):

    tensor = []

    for sen in corpus:
        token_ids = s.EncodeAsIds(sen)[:MAX_LENGTH]
        tensor.append(torch.tensor(token_ids, dtype=torch.long))

    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = f.readlines()

    word_index = {}
    index_word = {}

    for idx, line in enumerate(vocab):
        word = line.split("\t")[0]

        word_index.update({word: idx})
        index_word.update({idx: word})

    tensor = pad_sequence(
        tensor,
        batch_first=True,
        padding_value=s.pad_id(),
    )

    return tensor, word_index, index_word

In [157]:
practice_sentences = [
    "아버지가방에들어가신다.",
    "이 영화는 정말 재미있어요.",
    "별로였어요.",
]

practice_tensor, word_index, index_word = sp_tokenize(s, practice_sentences)

print("tensor shape:", practice_tensor.shape)
print(practice_tensor)
print("vocab size:", len(word_index))

first_ids = [
    token_id
    for token_id in practice_tensor[0].tolist()
    if token_id != s.pad_id()
]
print("첫 문장 복원:", s.DecodeIds(first_ids))
assert s.DecodeIds(first_ids) == practice_sentences[0]

tensor shape: torch.Size([3, 9])
tensor([[1406,   12,  387,   16, 1317,   12,  131,   19,    5],
        [  28,  127,   29, 1488,    5,    3,    3,    3,    3],
        [ 213, 2340,    5,    3,    3,    3,    3,    3,    3]])
vocab size: 8000
첫 문장 복원: 아버지가방에들어가신다.


### 5. `Dataset`과 `DataLoader`


```text
__len__()     → 데이터가 총 몇 개인지 반환
__getitem__() → i번째 (입력, 정답) 쌍 반환
DataLoader    → 여러 쌍을 mini-batch(미니배치)로 묶음
```




In [158]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# NSMC 문장을 token ID로 바꾼 뒤 Dataset에 전달
def make_loader(processor, data, vocab_path, shuffle):
    text_tensor, _, _ = sp_tokenize(
        processor,
        data["document"].tolist(),
        vocab_path,
    )
    label_tensor = torch.tensor(data["label"].values, dtype=torch.long)
    dataset = SimpleDataset(text_tensor, label_tensor)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)


unigram_train_loader = make_loader(s, train_data, "./korean_spm.vocab", True)
unigram_validation_loader = make_loader(s, validation_data, "./korean_spm.vocab", False)
unigram_test_loader = make_loader(s, test_data, "./korean_spm.vocab", False)

sample_batch, sample_labels = next(iter(unigram_train_loader))
print("문장 batch shape:", sample_batch.shape)
print("label batch shape:", sample_labels.shape)


문장 batch shape: torch.Size([512, 80])
label batch shape: torch.Size([512])


### 6. 가장 단순한 감정 분류기


```text
nn.Module 상속 → __init__에서 Layer 선언 → forward에서 데이터 흐름 작성
```


```text
token ID → Embedding → padding을 제외한 평균 → Linear → 긍정/부정
```

- `Embedding`: 각 token ID를 256개 숫자로 표현
- `Linear`: 256개 숫자를 부정 0 / 긍정 1의 두 점수로 변환
- `mask`와 평균: 여러 token vector를 문장 하나의 vector로 만듬
- `unsqueeze(-1)`: `[batch, 길이]` mask를 `[batch, 길이, 1]`로 변경
- `clamp(min=1)`: 빈 문장이 들어와도 0으로 나누지 않게 보호


In [159]:
class SimpleSentimentModel(nn.Module):
    def __init__(self, vocab_size, pad_id):
        super().__init__()

        self.pad_id = pad_id
        embedding_dim = 256  # 먼저 정의

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id,
        )
        self.output = nn.Linear(embedding_dim, 2)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)

        mask = (token_ids != self.pad_id).unsqueeze(-1)
        token_sum = (embedded * mask).sum(dim=1)
        token_count = mask.sum(dim=1).clamp(min=1)
        sentence_vector = token_sum / token_count

        return self.output(sentence_vector)


### 7. 모델 학습과 평가


```text
train_one_epoch() : 한 epoch 동안 weight(가중치) 업데이트
evaluate()        : weight를 바꾸지 않고 loss와 accuracy 계산
```

mini-batch의 다섯 단계

```text
1. pred = model(x)              forward(예측)
2. loss = loss_fn(pred, y)      오차 계산
3. optimizer.zero_grad()        이전 gradient 지우기
4. loss.backward()              이번 gradient 계산
5. optimizer.step()             weight 업데이트
```


In [160]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()                          # 학습 모드
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)  # 데이터도 GPU/CPU로

        pred = model(x)                    # 1. forward
        loss = loss_fn(pred, y)            # 2. loss

        optimizer.zero_grad()              # 3. gradient 초기화
        loss.backward()                    # 4. backward
        optimizer.step()                   # 5. parameter 업데이트

        total_loss += loss.item()

    return total_loss / len(loader)


#  7장의 evaluate()
def evaluate(model, loader, loss_fn, device):
    model.eval()                           # 평가 모드
    total_loss, correct = 0.0, 0

    with torch.no_grad():                  # gradient 추적 끄기
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


# 3 epoch 반복하고 validation 결과를 출력
def train_model(processor, train_loader, validation_loader, name):
    torch.manual_seed(SEED)
    model = SimpleSentimentModel(
        processor.get_piece_size(),
        processor.pad_id(),
    ).to(DEVICE)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=2e-3)
    validation_accuracy = 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, loss_fn, optimizer, DEVICE
        )
        validation_loss, validation_accuracy = evaluate(
            model, validation_loader, loss_fn, DEVICE
        )

        print(
            f"{name} epoch {epoch}: "
            f"train_loss={train_loss:.4f}, "
            f"validation_loss={validation_loss:.4f}, "
            f"validation accuracy={validation_accuracy:.3%}"
        )

    return model, validation_accuracy


In [161]:
unigram_model, unigram_validation_accuracy = train_model(
    s,
    unigram_train_loader,
    unigram_validation_loader,
    "Unigram-8000",
)

Unigram-8000 epoch 1: train_loss=0.4865, validation_loss=0.3840, validation accuracy=83.380%
Unigram-8000 epoch 2: train_loss=0.3576, validation_loss=0.3582, validation accuracy=84.840%
Unigram-8000 epoch 3: train_loss=0.3369, validation_loss=0.3589, validation accuracy=84.660%


### 8. `model_type=bpe`와 비교

이번에는 corpus와 `vocab_size=8000`은 그대로 두고 `--model_type=bpe`만 추가합니다. 즉, 성능 차이가 생기면 핵심 변경점은 Unigram과 BPE의 token 구성 방식입니다.

In [162]:
# LMS 학습 코드에서 model_prefix와 model_type만 변경합니다.
spm.SentencePieceTrainer.Train(
    '--input={} --model_prefix=korean_spm_bpe --vocab_size={} '
    '--model_type=bpe --pad_id=3 --minloglevel=2'.format(
        temp_file, vocab_size
    )
)

bpe_s = spm.SentencePieceProcessor()
bpe_s.Load('korean_spm_bpe.model')

print("Unigram:", s.EncodeAsPieces('아버지가방에들어가신다.'))
print("BPE:", bpe_s.EncodeAsPieces('아버지가방에들어가신다.'))

Unigram: ['▁아버지', '가', '방', '에', '들어', '가', '신', '다', '.']
BPE: ['▁아버', '지가', '방', '에', '들어', '가', '신', '다', '.']


In [163]:
bpe_train_loader = make_loader(
    bpe_s, train_data, "./korean_spm_bpe.vocab", True
)
bpe_validation_loader = make_loader(
    bpe_s, validation_data, "./korean_spm_bpe.vocab", False
)

bpe_test_loader = make_loader(
    bpe_s, test_data, "./korean_spm_bpe.vocab", False
)

bpe_model, bpe_validation_accuracy = train_model(
    bpe_s,
    bpe_train_loader,
    bpe_validation_loader,
    "BPE-8000",
)

BPE-8000 epoch 1: train_loss=0.4779, validation_loss=0.3736, validation accuracy=84.120%
BPE-8000 epoch 2: train_loss=0.3566, validation_loss=0.3535, validation accuracy=85.180%
BPE-8000 epoch 3: train_loss=0.3379, validation_loss=0.3538, validation accuracy=85.220%


### 9. 비교 결과와 최종 test accuracy

두 tokenizer는 validation accuracy로 비교합니다.


In [164]:
sample_texts = validation_data["document"].iloc[:1000]
unigram_mean_tokens = np.mean([len(s.EncodeAsIds(text)) for text in sample_texts])
bpe_mean_tokens = np.mean([len(bpe_s.EncodeAsIds(text)) for text in sample_texts])

comparison = pd.DataFrame({
    "tokenizer": ["Unigram-8000", "BPE-8000"],
    "평균 token 수": [unigram_mean_tokens, bpe_mean_tokens],
    "validation accuracy": [
        unigram_validation_accuracy,
        bpe_validation_accuracy,
    ],
})
display(comparison)

# LMS evaluate() 형태대로 loss와 accuracy를 함께 계산합니다.
# 모델 선택과 비교가 끝난 뒤 주 모델을 test data에서 마지막 한 번 평가합니다.
test_loss_fn = nn.CrossEntropyLoss()
test_loss, test_accuracy = evaluate(
    unigram_model,
    unigram_test_loader,
    test_loss_fn,
    DEVICE,
)

bpe_test_loss, bpe_test_accuracy = evaluate(
    bpe_model,
    bpe_test_loader,
    test_loss_fn,
    DEVICE,
)


print(f"Unigram-8000 test loss: {test_loss:.4f}")
print(f"Unigram-8000 test accuracy: {test_accuracy:.3%}")
print(f"BPE-8000 test loss: {bpe_test_loss:.4f}")
print(f"BPE-8000 test accuracy: {bpe_test_accuracy:.3%}")

,tokenizer,평균 token 수,validation accuracy
0,Unigram-8000,17.366,0.8466
1,BPE-8000,17.032,0.8522


Unigram-8000 test loss: 0.3718
Unigram-8000 test accuracy: 84.504%
BPE-8000 test loss: 0.3715
BPE-8000 test accuracy: 84.438%


## vocab_size를 수동 변경 반복 실행 결과
---
Unigram-4000 test loss: 0.3899  
Unigram-4000 test accuracy: 83.408%  
BPE-4000 test loss: 0.3914   
BPE-4000 test accuracy: 83.443%  

---
Unigram-8000 test loss: 0.3718  
Unigram-8000 test accuracy: 84.504%  
BPE-8000 test loss: 0.3715  
BPE-8000 test accuracy: 84.438%  

---
Unigram-12000 test loss: 0.3667  
Unigram-12000 test accuracy: 84.849%  
BPE-12000 test loss: 0.3668  
BPE-12000 test accuracy: 84.820%  

---
Unigram-20000 test loss: 0.3690  
Unigram-20000 test accuracy: 84.825%  
BPE-20000 test loss: 0.3710  
BPE-20000 test accuracy: 84.674%  

---
Unigram-30000 test loss: 0.3706  
Unigram-30000 test accuracy: 84.847%  
BPE-30000 test loss: 0.3744  
BPE-30000 test accuracy: 84.763%  

---

vocab_size 12000 이후 test accuracy 변화가 정체되는 것을 확인
unigreen이 BPE에 비해 미세하게 우세한 것을 확인

## 결과 해석

1. `Unigram-8000`의 validation accuracy는 **84.66%**, `BPE-8000`은 **85.220%**였다.
2. 평균 token 수는 Unigram이 **17.366개**, BPE가 **17.032개**였다.



> NSMC train 문장을 SentencePiece에 주어 8,000개의 subword vocabulary를 학습했습니다. 문장을 token ID로 바꾸고 padding한 뒤 각 token embedding의 평균을 이용해 긍정과 부정을 분류했습니다. Unigram과 BPE를 같은 조건으로 비교했고, 주 모델의 test accuracy가 80%를 넘는지 확인했습니다.


### 코드에 사용된 개념들을 언제 어디서 배웠는가?

| 코드 내용 | LMS 노드 | 적용 방법 |
|---|---|---|
| SentencePiece 학습·사용 | 현재 프로젝트 Step 2~3 | NSMC corpus, PAD 분리, UTF-8 |
| `SimpleDataset`, `DataLoader` | 「PyTorch와 텐서 첫걸음」 6장 | 입력을 영화 리뷰 token ID로 변경 |
| `nn.Module`, `forward`, `nn.Linear` | 「PyTorch와 텐서 첫걸음」 5장 | 앞에 `Embedding`과 token 평균 추가 |
| `CrossEntropyLoss`, Optimizer | 「PyTorch와 텐서 첫걸음」 5장 | 기존 검증값을 보존해 `AdamW`, `lr=2e-3` 유지 |
| `train_one_epoch`, `evaluate` | 「PyTorch와 텐서 첫걸음」 7장 | 이미지 대신 리뷰 token ID 사용 |